In [15]:
!pip install langgraph

In [16]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [17]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [18]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(60)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [19]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [20]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    config = {"configurable": {"thread_id": 'thread-1'}}
    graph.invoke({"input": "start"}, config=config)
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [21]:
graph.get_state(config)

StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f838-67e8-66ef-8001-ad8fc1092bbb'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-08-02T09:31:53.632312+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f838-67e5-63a7-8000-c9b854c9e955'}}, tasks=(PregelTask(id='defbec41-04bc-0cf9-e71f-914cdab15933', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [22]:
list(graph.get_state_history(config))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f838-67e8-66ef-8001-ad8fc1092bbb'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-08-02T09:31:53.632312+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f838-67e5-63a7-8000-c9b854c9e955'}}, tasks=(PregelTask(id='defbec41-04bc-0cf9-e71f-914cdab15933', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start'}, next=('step_1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f838-67e5-63a7-8000-c9b854c9e955'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2025-08-02T09:31:53.630999+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'ch

In [23]:
graph.invoke(None, config=config)

⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed


{'input': 'start', 'step1': 'done', 'step2': 'done'}

In [24]:
graph.get_state(config)

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f83b-d44e-6603-8003-97a7d5a227d4'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2025-08-02T09:33:25.529326+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f83b-d44b-6fd7-8002-c41fa8173c74'}}, tasks=(), interrupts=())

### Time Travel

In [28]:
graph.get_state({'configurable': {"thread_id": 'thread-1', 'checkpoint_id': "1f06f838-67e5-63a7-8000-c9b854c9e955"}})

StateSnapshot(values={'input': 'start'}, next=('step_1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_id': '1f06f838-67e5-63a7-8000-c9b854c9e955'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2025-08-02T09:31:53.630999+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f838-67e0-6d93-bfff-2b12c92f6152'}}, tasks=(PregelTask(id='263484c9-4937-400e-3867-29bb1d75d5ad', name='step_1', path=('__pregel_pull', 'step_1'), error=None, interrupts=(), state=None, result={'step1': 'done', 'input': 'start'}),), interrupts=())

In [29]:
graph.invoke(None, {'configurable': {"thread_id": 'thread-1', 'checkpoint_id': "1f06f838-67e5-63a7-8000-c9b854c9e955"}})

✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed


{'input': 'start', 'step1': 'done', 'step2': 'done'}

In [31]:
list(graph.get_state_history(config))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f83f-d75d-6fd7-8003-991bc4d4aa90'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2025-08-02T09:35:13.224473+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f83f-d75b-6055-8002-31b3ef0c302f'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f83f-d75b-6055-8002-31b3ef0c302f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-08-02T09:35:13.223211+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f06f83d-9b24-615e-8001-4f58f9b955c7'}}, tasks=(PregelTask(id='229e7140-b4c6-1eb4-9b5f-cc7508f2a23

In [ ]:
graph.update_state({"configurable": {"thread_id": 'thread-1', 'checkpoint_id': "1f06f838-67e5-63a7-8000-c9b854c9e955", "checkpoint_ns": ""}}, {'topic':'samosa'})